# 9.4 잠재요인 협업 필터링
## 확률적 경사하강법을 이용한 행렬 분해

In [6]:
import numpy as np

# 원본 행렬 R 생성, 분해 행렬 P와 Q 초기화, 잠재요인 차원 K는 3 설정.
R = np.array([[4, np.nan, np.nan, 2, np.nan ],
              [np.nan, 5, np.nan, 3, 1 ],
              [np.nan, np.nan, 3, 4, 4 ],
              [5, 2, 1, 2, np.nan ]])
num_users, num_items = R.shape
K=3

# P와 Q 매트릭스의 크기를 지정 / 정규분포를 가진 random값 입력

np.random.seed(1)
P = np.random.normal(scale=1./K, size=(num_users, K))
Q = np.random.normal(scale=1./K, size=(num_items, K))

In [7]:
from sklearn.metrics import mean_squared_error

def get_rmse(R, P, Q, non_zeros):
    error = 0
    # P * Q.T  > 예측 R 행렬 생성
    full_pred_matrix = np.dot(P, Q.T)

    # 실제 R 행렬에서 널이 아닌 값의 위치 인덱스 추출 > 실제 R 행렬과 예측 행렬의 RMSE 추출
    x_non_zero_ind = [non_zero[0] for non_zero in non_zeros]
    y_non_zero_ind = [non_zero[1] for non_zero in non_zeros]
    R_non_zeros = R[x_non_zero_ind, y_non_zero_ind]
    full_pred_matrix_non_zeros = full_pred_matrix[x_non_zero_ind, y_non_zero_ind]

    mse = mean_squared_error(R_non_zeros, full_pred_matrix_non_zeros)
    rmse = np.sqrt(mse)

    return rmse

In [8]:
# R > 0 인 행 위치, 열 위치, 값을 non_zeros 리스트에 저장.
non_zeros = [ (i, j, R[i,j]) for i in range(num_users) for j in range(num_items) if R[i,j] > 0 ]

steps=1000
learning_rate=0.01
r_lambda=0.01

# SGD : P와 Q 계속 업데이트
for step in range(steps):
    for i, j, r in non_zeros:
        # 실제 값 - 예측 값
        eij = r - np.dot(P[i, :], Q[j, :].T)
        # Regularization을 반영한 SGD 업데이트 공식 적용
        P[i,:] = P[i,:] + learning_rate*(eij * Q[j, :] - r_lambda*P[i,:])
        Q[j,:] = Q[j,:] + learning_rate*(eij * P[i, :] - r_lambda*Q[j,:])

    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 50) == 0 :
        print("### iteration step : ", step," rmse : ", rmse)

### iteration step :  0  rmse :  3.2388050277987723
### iteration step :  50  rmse :  0.4876723101369648
### iteration step :  100  rmse :  0.1564340384819247
### iteration step :  150  rmse :  0.07455141311978046
### iteration step :  200  rmse :  0.04325226798579314
### iteration step :  250  rmse :  0.029248328780878973
### iteration step :  300  rmse :  0.022621116143829466
### iteration step :  350  rmse :  0.019493636196525135
### iteration step :  400  rmse :  0.018022719092132704
### iteration step :  450  rmse :  0.01731968595344266
### iteration step :  500  rmse :  0.016973657887570753
### iteration step :  550  rmse :  0.016796804595895633
### iteration step :  600  rmse :  0.01670132290188466
### iteration step :  650  rmse :  0.01664473691247669
### iteration step :  700  rmse :  0.016605910068210026
### iteration step :  750  rmse :  0.016574200475705
### iteration step :  800  rmse :  0.01654431582921597
### iteration step :  850  rmse :  0.01651375177473524
### iterati

In [9]:
pred_matrix = np.dot(P, Q.T)
print('예측 행렬:\n', np.round(pred_matrix, 3))

예측 행렬:
 [[3.991 0.897 1.306 2.002 1.663]
 [6.696 4.978 0.979 2.981 1.003]
 [6.677 0.391 2.987 3.977 3.986]
 [4.968 2.005 1.006 2.017 1.14 ]]


# 9.8 파이썬 추천 시스템 패키지 - Surprise


In [10]:
pip install surprise

In [11]:
!pip install numpy==1.26.4 --upgrade --force-reinstall

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


## Surprise를 이용한 추천 시스템 구축

In [1]:
# 관련 모듈 임포트
from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import train_test_split


- 사용 데이터셋 - movieLens사이트 제공 과거 버전의 데이터셋
- 로우 레벨의 칼럼 데이터를 칼럼 레벨의 데이터로 자체 변경하므로 원본인 로우 레벨의 사용자-아이템 평점 데이터를 데이터셋으로 적용해야함.

In [2]:
data = Dataset.load_builtin('ml-100k')
# 수행 시마다 동일하게 데이터를 분할하기 위해 random_state 값 부여
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

In [3]:
algo = SVD(random_state=0)
algo.fit(trainset)

- SVD객체.test() 반환 결과 : 입력인자 데이터셋 크기와 같은 파이썬 리스트
- Prediction 객체 > surprise 패키지 제공 데이터 타입
  * uid(user id), iid(movie/item id),r_ui(실제평점) 기반해 예측한 예측평점(est)를 튜플형태로 가짐
  * .details - 추천예측 안되는 경우 로그용 데이터 남김
  * was_impossible = True : 예측값 생성할 수 없는 데이터

In [4]:
predictions = algo.test( testset )
print('prediction type :',type(predictions), ' size:',len(predictions))
print('prediction 결과의 최초 5개 추출')
predictions[:5]

prediction type : <class 'list'>  size: 25000
prediction 결과의 최초 5개 추출


[Prediction(uid='120', iid='282', r_ui=4.0, est=3.5114147666251547, details={'was_impossible': False}),
 Prediction(uid='882', iid='291', r_ui=4.0, est=3.573872419581491, details={'was_impossible': False}),
 Prediction(uid='535', iid='507', r_ui=5.0, est=4.033583485472447, details={'was_impossible': False}),
 Prediction(uid='697', iid='244', r_ui=5.0, est=3.8463639495936905, details={'was_impossible': False}),
 Prediction(uid='751', iid='385', r_ui=4.0, est=3.1807542478219157, details={'was_impossible': False})]

**속성 추출**
- predict객체.uid
- predict객체.iid
- predict객체.est

In [5]:
[ (pred.uid, pred.iid, pred.est) for pred in predictions[:3] ]

[('120', '282', 3.5114147666251547),
 ('882', '291', 3.573872419581491),
 ('535', '507', 4.033583485472447)]

- test가 아닌 predict()로 추천 예측 가능
- predict() : 개별 사용자의 아이템에 대한 추천 평점 예측
- input(문자열) : uid, iid (r_ui는 선택)

In [6]:
# 사용자 아이디, 아이템 아이디는 문자열로 입력해야 함.
uid = str(196)
iid = str(302)
pred = algo.predict(uid, iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}


- accuracy 모듈은 RMSE, MSE 등으로 추천 시스템의 성능 평가 정보 제공

In [7]:
accuracy.rmse(predictions)

RMSE: 0.9467


0.9466860806937948

## Surprise 주요 모듈 소개
### Dataset
1. Dataset.load_builtin
  * 무비렌즈 아카이브 FTP 서버에서 무비렌즈 데이터 내려받음
  * name = ml-100k(default)/ml-1M 내려받을 수 있음
2. Dataset.load_from_file
  * OS 파일에서 데이터를 로딩할 때 사용
  * 콤마/탭으로 구분된 포맷의 OS파일에서 데이터 로딩
  * parameter - OS 파일명 , Reader로 파일의 포맷 지정
3. Dataset.load_from_df
  * 판다스의 DataFrame에서 데이터 로딩
  * DataFrame은 반드시 3개의 칼럼인 사용자 아이디, 아이템 아이디, 평점 순으로 칼럼 순서가 정해져있어야 함.
  * parameter - DataFrame 객체, Rader로 파일의 포맷 지정
⚠️Surprise > OS파일 로딩 시 주의 점 : 칼럼명 헤더 문자열 있으면 안됨

In [8]:
import pandas as pd

ratings = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings.csv')
# ratings_noh.csv 파일로 unload 시 index 와 header를 모두 제거한 새로운 파일 생성.
ratings.to_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings_noh.csv', index=False, header=False)

- Reader 클래스 : ratings_noh.csv 파일 파싱 포맷 정의/파싱 정보 알려줌
  * 생성자에 각 필드의 칼럼명,구분자, 최소~최대 평점 입력 ➡ 객체 생성
  * line_format : 칼럼 user item rating timestamp 명시, 문자열을 공백으로 구분
  * sep : 구분자 명시
  * rating_scale : 평점 단위 = 0.5, 최대 평점 = 5
- Dataset.load_from_file() : Reader 객체 참조 데이터 파일 파싱하며 로딩

In [14]:
from surprise import Reader

reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5))
data=Dataset.load_from_file('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings_noh.csv',reader=reader)

In [10]:
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

# 수행시마다 동일한 결과 도출을 위해 random_state 설정
algo = SVD(n_factors=50, random_state=0)

# 학습/예측/평가
algo.fit(trainset)
predictions = algo.test( testset )
accuracy.rmse(predictions)

RMSE: 0.8708


0.8708344753692029

### 판다스 DataFrame에서 Surprise 데이터 세트로 로딩
- Dataset.load_from_df()

In [11]:
import pandas as pd
from surprise import Reader, Dataset

ratings = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings.csv')
reader = Reader(rating_scale=(0.5, 5.0))



In [17]:
ratings

,1,1.1,4.0,964982703
0,1,3,4.0,964981247
1,1,6,4.0,964982224
2,1,47,5.0,964983815
3,1,50,5.0,964982931
4,1,70,3.0,964982400
...,...,...,...,...
100830,610,166534,4.0,1493848402
100831,610,168248,5.0,1493850091
100832,610,168250,5.0,1494273047
100833,610,168252,5.0,1493846352


In [19]:
# ratings DataFrame 에서 컬럼은 사용자 아이디, 아이템 아이디, 평점 순서를 지켜야 합니다.
ratings.columns = ['userId', 'movieId', 'rating','timestamp']

data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)
predictions = algo.test( testset )
accuracy.rmse(predictions)

RMSE: 0.8708


0.8708344753692029

## Surprise 추천 알고리즘 클래스

1.  SVD - 행렬 분해, 잠재 요인 협업 필터링
  * 사용자 baseline 편향성 감안한 쳥점 예측에 Regularization 적용
  * 사용자 예측 rating $\hat{r}_{ui}=\mu+bu+bi+aTipu$
  * Regularization 적용 비용함수 : $\sum{({r}_{ui}-\hat{r}_{ui})2}+\lambda(b2i+b2u+||qi||2+||pu||2)$
  * **parameters**
     1. n_factors : 잠재요인 K 수 (default-100), 클수록 과적합 위험 ⬆, 정확도 ⬆
     2. n_epochs : SGD 수행시 반복 횟수(default - 20)
     3. biaseed - 베이스라인 사용자 편향 적용 여부 (default-True)
2.  KNNBasic - 최근접 이웃 협업 필터링 KNN알고리즘
3.  BaselineOnly - 사용자/아이템 Bias를 감안한 SGD베이스라인 알고리즘
4. 이외 알고리즘 - SVD++, NMF, Slpe One, Co-Clustering/ Baseline - 각 개인이 평점을 부여하는 성향 반영해 평점 계산

## 베이스라인 평점
- 개인의 성향을 반영해 ㅐ아이템 평가에 편향성 요소를 반영해 평점 부과
- 전체 평균 평점 + 사용자 편향 점수 + 아이템 편향 점수
- 전체 평균 평점 = 모든 사용자 아이템 평점 평균값
- 사용자 편향 점수 = 사용자별 아이템 평균값 - 전체 평균 평점
- 아이템 편향 점수 = 아이템별 평점 평균 값 - 전체 평균 평점

## 교차검증과 하이퍼 파라미터 튜닝
- suprise.modelsection - cross_validate()와 GridSearchCV클래스 제공

In [22]:
from surprise.model_selection import cross_validate

ratings = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings.csv') # reading data in pandas df
ratings.columns = ['userId', 'movieId', 'rating','timestamp']
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

algo = SVD(random_state=0)
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8717  0.8712  0.8723  0.8724  0.8815  0.8738  0.0039  
MAE (testset)     0.6696  0.6694  0.6696  0.6680  0.6798  0.6713  0.0043  
Fit time          2.28    1.39    1.45    1.39    1.51    1.61    0.34    
Test time         0.18    0.25    0.11    0.27    0.10    0.18    0.07    


{'test_rmse': array([0.87171692, 0.87117491, 0.87225411, 0.87238817, 0.88150963]),
 'test_mae': array([0.66958695, 0.66941221, 0.66962236, 0.66796536, 0.67984648]),
 'fit_time': (2.2812085151672363,
  1.3924455642700195,
  1.449477195739746,
  1.390857458114624,
  1.5147252082824707),
 'test_time': (0.1770644187927246,
  0.2455298900604248,
  0.1055753231048584,
  0.2739133834838867,
  0.10271763801574707)}

In [23]:
from surprise.model_selection import GridSearchCV

param_grid = {'n_epochs': [20, 40, 60], 'n_factors': [50, 100, 200] }

# CV = 3, 성능 평가 = rmse, mse
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

# 최고 RMSE Evaluation 점수와 그때의 하이퍼 파라미터
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8766244571341707
{'n_epochs': 20, 'n_factors': 50}


## Surprise를 이용한 개인화 영화 추천 시스템 구축

In [24]:
# 다음 코드는 train_test_split( )으로 분리되지 않는 데이터 세트에 fit( )을 호출해 오류가 발생합니다.
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
algo = SVD(n_factors=50, random_state=0)
algo.fit(data)

AttributeError: 'DatasetAutoFolds' object has no attribute 'n_users'

In [26]:
from surprise.dataset import DatasetAutoFolds

reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5))
# DatasetAutoFolds 클래스를 ratings_noh.csv 파일 기반으로 생성.
data_folds = DatasetAutoFolds(ratings_file='/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_ratings_noh.csv', reader=reader)

#전체 데이터를 학습데이터로 생성함.
trainset = data_folds.build_full_trainset()

In [27]:
algo = SVD(n_epochs=20, n_factors=50, random_state=0)
algo.fit(trainset)

In [30]:
# 영화에 대한 상세 속성 정보 DataFrame로딩
movies = pd.read_csv('/content/drive/MyDrive/Sample_data/ml-latest-small/ml_latest_small_movies.csv')

# userId=9 의 movieId 데이터 추출하여 movieId=42 데이터가 있는지 확인.
movieIds = ratings[ratings['userId']==9]['movieId']
if movieIds[movieIds==42].count() == 0:
    print('사용자 아이디 9는 영화 아이디 42의 평점 없음')

print(movies[movies['movieId']==42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
    movieId                   title              genres
38       42  Dead Presidents (1995)  Action|Crime|Drama


In [31]:
uid = str(9)
iid = str(42)

pred = algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 3.10   {'was_impossible': False}


In [32]:
def get_unseen_surprise(ratings, movies, userId):
    #입력값으로 들어온 userId에 해당하는 사용자가 평점을 매긴 모든 영화를 리스트로 생성
    seen_movies = ratings[ratings['userId']== userId]['movieId'].tolist()

    # 모든 영화들의 movieId를 리스트로 생성.
    total_movies = movies['movieId'].tolist()

    # 모든 영화들의 movieId중 이미 평점을 매긴 영화의 movieId를 제외하여 리스트로 생성
    unseen_movies= [movie for movie in total_movies if movie not in seen_movies]
    print('평점 매긴 영화수:',len(seen_movies), '추천대상 영화수:',len(unseen_movies), \
          '전체 영화수:',len(total_movies))

    return unseen_movies

unseen_movies = get_unseen_surprise(ratings, movies, 9)

평점 매긴 영화수: 46 추천대상 영화수: 9696 전체 영화수: 9742


In [33]:
def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):
    # 알고리즘 객체의 predict() 메서드를 평점이 없는 영화에 반복 수행한 후 결과를 list 객체로 저장
    predictions = [algo.predict(str(userId), str(movieId)) for movieId in unseen_movies]

    # predictions list 객체는 surprise의 Predictions 객체를 원소로 가지고 있음.
    # [Prediction(uid='9', iid='1', est=3.69), Prediction(uid='9', iid='2', est=2.98),,,,]
    # 이를 est 값으로 정렬하기 위해서 아래의 sortkey_est 함수를 정의함.

    # sortkey_est 함수 : list 객체의 sort() 함수의 키 값으로 사용되됨. 정렬 수행.
    def sortkey_est(pred):
        return pred.est

    # sortkey_est( ) 반환값의 내림 차순으로 정렬 수행 > top_n 최상위 값 추출.
    predictions.sort(key=sortkey_est, reverse=True)
    top_predictions= predictions[:top_n]

    # top_n으로 추출된 영화의 정보 추 >. 영화 아이디, 추천 예상 평점, 제목
    top_movie_ids = [ int(pred.iid) for pred in top_predictions]
    top_movie_rating = [ pred.est for pred in top_predictions]
    top_movie_titles = movies[movies.movieId.isin(top_movie_ids)]['title']
    top_movie_preds = [ (id, title, rating) for id, title, rating in zip(top_movie_ids, top_movie_titles, top_movie_rating)]

    return top_movie_preds

unseen_movies = get_unseen_surprise(ratings, movies, 9)
top_movie_preds = recomm_movie_by_surprise(algo, 9, unseen_movies, top_n=10)
print('##### Top-10 추천 영화 리스트 #####')

for top_movie in top_movie_preds:
    print(top_movie[1], ":", top_movie[2])

평점 매긴 영화수: 46 추천대상 영화수: 9696 전체 영화수: 9742
##### Top-10 추천 영화 리스트 #####
Usual Suspects, The (1995) : 4.2267942523743605
Star Wars: Episode IV - A New Hope (1977) : 4.211016328181448
Shawshank Redemption, The (1994) : 4.197037380337041
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964) : 4.142771982890766
Godfather, The (1972) : 4.136604760890598
Reservoir Dogs (1992) : 4.124673084995863
Streetcar Named Desire, A (1951) : 4.120523998974084
Goodfellas (1990) : 4.069435046627849
Glory (1989) : 4.067497992056492
All the President's Men (1976) : 4.062778702133245
